# Huấn luyện Ablation DeepVQE: B1a & B1b (PReLU)
Bản notebook này dùng để train các biến thể B1a (Shared PReLU) và B1b (Per-channel PReLU) theo lộ trình Ablation.

WandB Integration, Gradient Norm Tracking, Periodic PESQ/STOI Evaluation, Batch Size 8 URL: https://drive.google.com/drive/folders/15NiTQus97k9YGpyPkRCTXqpt-O--VHGA?usp=sharing


## 1. Cài đặt môi trường & Kết nối Kaggle Working Dir

In [ ]:
!pip install wandb gdown matplotlib torchaudio tensorboard soundfile pandas tqdm einops pesq pystoi


import os
WORK_DIR = '/kaggle/working/DeepVQE_Workspace'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Cấu hình lưu checkpoint vào thư mục làm việc
CHECKPOINT_DIR = os.path.join(WORK_DIR, 'checkpoint_B1')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Thư mục làm việc hiện tại: {os.getcwd()}")

## 2. Clone mã nguồn DeepVQE

In [ ]:
# Repo này có sẵn deepvqe.py, ablation/ và stream/ benchmark scripts.
GIT_REPO = "https://github.com/hoxuanphu/Pdeepvqe.git"
REPO_DIR = "deepvqe"

import os
if not os.path.exists(REPO_DIR):
    !git clone {GIT_REPO} {REPO_DIR}
else:
    print(f"Thư mục {REPO_DIR} đã tồn tại. Đang cập nhật code mới nhất từ GitHub...")
    !cd {REPO_DIR} && git pull origin main

os.chdir(REPO_DIR)
!pip install -q pesq pystoi torchmetrics
print(f"Đã vào thư mục mã nguồn: {os.getcwd()}")


## 3. Tải bộ dữ liệu VoiceBank-DEMAND
Dữ liệu gốc được tải từ Đại học Edinburgh (chiếm khoảng 4GB - 5GB sau khi giải nén).

**Lưu ý**: VCTK-DEMAND gốc có sample rate 48kHz. Code training sẽ tự resample xuống 16kHz.

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

# Thư mục chứa file ZIP trên Google Drive
drive_data_dir = "/kaggle/working/DeepVQE_Workspace/data/voicebank-demand"
os.makedirs(drive_data_dir, exist_ok=True)

# Thư mục trên máy ảo Colab (Local SSD) để giải nén và train cho nhanh
data_dir = "/kaggle/working/data/voicebank-demand"
os.makedirs(data_dir, exist_ok=True)

datasets = [
    ("clean_trainset_28spk_wav.zip", "https://drive.google.com/file/d/1NJr2O4Ik6ueSFlIGSvub8dnFXGTHJ2PG/view?usp=sharing"),
    ("noisy_trainset_28spk_wav.zip", "https://drive.google.com/file/d/1OqpDIvpVyaTnMbwY1Qt__hfX3X4siMtU/view?usp=sharing"),
    ("clean_testset_wav.zip", "https://drive.google.com/file/d/1GQc-T1R4FNrhRjTn7AAvAenZTIQEazeH/view?usp=sharing"),
    ("noisy_testset_wav.zip", "https://drive.google.com/file/d/1rimmCqxXRYRIXZcPkGjQiacr6j1QsMAH/view?usp=sharing")
]

for filename, url in datasets:
    drive_zip = os.path.join(drive_data_dir, filename)
    local_zip = os.path.join(data_dir, filename)
    extract_folder = local_zip.replace(".zip", "")
    
    # 1. Ưu tiên kiểm tra trên Google Drive trước
    if os.path.exists(drive_zip):
        print(f"✅ Đã tìm thấy {filename} trên Google Drive!")
        if not os.path.exists(local_zip) and not os.path.exists(extract_folder):
            print(f"Đang copy {filename} từ Drive sang Local SSD...")
            shutil.copy2(drive_zip, local_zip)
    else:
        # 2. Không có trên Drive -> Tải trực tiếp từ web xuống Local SSD
        print(f"🌐 Không thấy {filename} trên Drive, đang tải trực tiếp từ web xuống Local SSD...")
        if not os.path.exists(extract_folder):
            if os.path.exists(local_zip) and not zipfile.is_zipfile(local_zip):
                print(f"File {local_zip} bị lỗi (không phải ZIP), đang xóa...")
                os.remove(local_zip)
            if not os.path.exists(local_zip):
                os.system(f"gdown --fuzzy {url} -O {local_zip}")
    
    # 3. Giải nén trên Local SSD
    if not os.path.exists(extract_folder):
        if os.path.exists(local_zip):
            print(f"Đang giải nén {filename}...")
            with zipfile.ZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            print(f"Hoàn tất giải nén {filename}.")
        else:
            print(f"❌ Lỗi: Không tìm thấy file {local_zip} để giải nén!")



## 4. Xử lý và phân chia Dataset (Tạo file CSV)
Tạo các file metadata dạng CSV (`train.csv`, `valid.csv`, `test.csv`).

In [ ]:
import glob
import pandas as pd

def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(os.path.join(clean_dir, "*.wav")))
    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    
    assert len(clean_files) == len(noisy_files), f"Số lượng file không khớp! ({len(clean_files)} != {len(noisy_files)})"
    mismatched = [(os.path.basename(c), os.path.basename(n))
                  for c, n in zip(clean_files, noisy_files)
                  if os.path.basename(c) != os.path.basename(n)]
    assert not mismatched, f"Tên file clean/noisy không khớp: {mismatched[:5]}"
    
    data = []
    for c, n in zip(clean_files, noisy_files):
        filename = os.path.basename(c)
        data.append({
            "ID": filename.replace(".wav", ""),
            "clean_wav": os.path.abspath(c),
            "noisy_wav": os.path.abspath(n)
        })
        
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Đã tạo {output_csv} với {len(df)} mẫu.")

base_dir = data_dir
# Tạo file CSV thô
create_csv(f"{base_dir}/clean_trainset_28spk_wav", f"{base_dir}/noisy_trainset_28spk_wav", f"{base_dir}/train_full.csv")
create_csv(f"{base_dir}/clean_testset_wav", f"{base_dir}/noisy_testset_wav", f"{base_dir}/test.csv")

# Tách train_full thành train (90%) và valid (10%)
df_train_full = pd.read_csv(f"{base_dir}/train_full.csv")
df_train_full = df_train_full.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df_train_full) * 0.90)
df_train_full.iloc[:split_idx].to_csv(f"{base_dir}/train.csv", index=False)
df_train_full.iloc[split_idx:].to_csv(f"{base_dir}/valid.csv", index=False)

print(f"Train: {split_idx} | Valid: {len(df_train_full) - split_idx}")
print("Hoàn tất tạo metadata (train.csv, valid.csv, test.csv)!")

## 5. Cấu hình Hyperparameters

In [ ]:
CONFIG = {
    'seed': 1234,
    'sample_rate': 16000,
    'n_fft': 512,
    'hop_length': 256,
    'win_length': 512,
    'stft_window': 'sqrt_hann',   
    
    'batch_size': 8,              
    'valid_batch_size': 4,        
    'grad_accum_steps': 1,        
    'num_workers': 2,
    'persistent_workers': True,
    'prefetch_factor': 2,
    'progress_update_every': 10,  
    'epochs': 100,
    'lr': 1e-4,
    'grad_clip': 5.0,
    
    'lamda_ri': 30,
    'lamda_mag': 70,
    'compress_factor': 0.3,
    
    'train_csv': f'{data_dir}/train.csv',
    'valid_csv': f'{data_dir}/valid.csv',
    'test_csv': f'{data_dir}/test.csv',
    
    'checkpoint_dir': f'{WORK_DIR}/checkpoint_B1',
    'output_dir': f'{WORK_DIR}/checkpoint_B1',
    'resume_checkpoint': 'last.pt',   # FIX2: resume từ last để không mất progress
    
    'scheduler_factor': 0.5,
    'scheduler_patience': 5,
    'scheduler_min_lr': 1e-6,
    
    'early_stopping_patience': 15,
    
    'use_amp': True,
    
    'augment': True,
    'aug_gain_range_db': (-6, 6),
    'aug_snr_remix_range': (0, 20),
    'aug_prob': 0.5,
    
    'use_multi_gpu': True,   # Tùy chọn chạy 2 GPU (Kaggle T4x2)
    'tensorboard': False,
    'use_wandb': True,
    'wandb_project': 'DeepVQE-Ablation-B1',
    'eval_pesq_every': 5,
    'eval_pesq_samples': 50,
    'log_audio_every': 5,
}

print('Config:', CONFIG)

## 6. Dataset & DataLoader (Pure PyTorch)
★ V2: Thêm Data Augmentation (Random Gain, SNR Remix, Speed Perturbation) cho training set.

In [ ]:
import random
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG['seed'])


class VCTKDemandDataset(Dataset):
    """Dataset cho VCTK-DEMAND: đọc cặp (noisy, clean) từ CSV.
    ★ V2: hỗ trợ data augmentation cho training."""
    
    def __init__(self, csv_path, sample_rate=16000, segment_len=None, augment_cfg=None):
        self.df = pd.read_csv(csv_path)
        self.sample_rate = sample_rate
        self.segment_len = segment_len  # Số sample cắt mỗi đoạn (None = giữ nguyên)
        self.aug = augment_cfg          # ★ V2: None = no augmentation
    
    def __len__(self):
        return len(self.df)
    
    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        return wav.squeeze(0)  # [T]
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        noisy = self._load(row['noisy_wav'])
        clean = self._load(row['clean_wav'])
        
        # Cắt về cùng chiều dài
        min_len = min(noisy.shape[0], clean.shape[0])
        noisy = noisy[:min_len]
        clean = clean[:min_len]
        
        # Random crop nếu đặt segment_len
        if self.segment_len is not None and min_len > self.segment_len:
            start = random.randint(0, min_len - self.segment_len)
            noisy = noisy[start:start + self.segment_len]
            clean = clean[start:start + self.segment_len]
        
        # ★ V2: Data Augmentation (chỉ khi training)
        if self.aug is not None:
            aug_prob = self.aug.get('aug_prob', 0.5)
            
            # 1. Random Gain — thay đổi biên độ cả cặp (noisy, clean) cùng lúc
            if random.random() < aug_prob:
                lo, hi = self.aug['aug_gain_range_db']
                gain_db = random.uniform(lo, hi)
                gain = 10 ** (gain_db / 20)
                noisy = noisy * gain
                clean = clean * gain
            
            # 2. SNR Remix — tách noise = noisy - clean, trộn lại với SNR mới
            if random.random() < aug_prob:
                noise = noisy - clean
                lo, hi = self.aug['aug_snr_remix_range']
                target_snr = random.uniform(lo, hi)
                clean_rms = clean.pow(2).mean().sqrt().clamp(min=1e-8)
                noise_rms = noise.pow(2).mean().sqrt().clamp(min=1e-8)
                target_noise_rms = clean_rms / (10 ** (target_snr / 20))
                noisy = clean + noise * (target_noise_rms / noise_rms)
            
        return {'noisy': noisy, 'clean': clean, 'id': row['ID']}


def collate_fn(batch):
    """Pad tất cả wav trong batch về cùng chiều dài (max len)."""
    max_len = max(item['noisy'].shape[0] for item in batch)
    
    noisy_batch = []
    clean_batch = []
    ids = []
    for item in batch:
        n, c = item['noisy'], item['clean']
        pad_len = max_len - n.shape[0]
        if pad_len > 0:
            n = torch.nn.functional.pad(n, (0, pad_len))
            c = torch.nn.functional.pad(c, (0, pad_len))
        noisy_batch.append(n)
        clean_batch.append(c)
        ids.append(item['id'])
    
    return {
        'noisy': torch.stack(noisy_batch),   # [B, T]
        'clean': torch.stack(clean_batch),   # [B, T]
        'id': ids,
    }


# Tạo DataLoader
# Dùng segment_len = 3s * 16000 = 48000 sample cho training để tiết kiệm VRAM
# ★ V2: augment_cfg chỉ truyền cho train, valid giữ nguyên
aug_cfg = {k: v for k, v in CONFIG.items() if k.startswith('aug_')} if CONFIG.get('augment') else None

train_dataset = VCTKDemandDataset(CONFIG['train_csv'], CONFIG['sample_rate'], segment_len=48000, augment_cfg=aug_cfg)
valid_dataset = VCTKDemandDataset(CONFIG['valid_csv'], CONFIG['sample_rate'], segment_len=48000)  # FIX3: giới hạn 3s để tránh OOM khi pad batch

loader_kwargs = dict(
    num_workers=CONFIG['num_workers'],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if CONFIG['num_workers'] > 0:
    loader_kwargs.update(
        persistent_workers=CONFIG.get('persistent_workers', False),
        prefetch_factor=CONFIG.get('prefetch_factor', 2),
    )

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
    drop_last=True, **loader_kwargs
)
valid_loader = DataLoader(
    valid_dataset, batch_size=CONFIG.get('valid_batch_size', CONFIG['batch_size']), shuffle=False,
    drop_last=False, **loader_kwargs
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Valid: {len(valid_dataset)} samples, {len(valid_loader)} batches')
if aug_cfg:
    print(f'Augmentation: ON (gain={aug_cfg["aug_gain_range_db"]}, snr_remix={aug_cfg["aug_snr_remix_range"]}, prob={aug_cfg["aug_prob"]})')
else:
    print('Augmentation: OFF')

## 7. Khởi tạo Model, Optimizer, Scheduler

In [ ]:
import sys
import time
import json
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

# Đảm bảo import được deepvqe.py
work_dir = '/kaggle/working/DeepVQE_Workspace/deepvqe'
if work_dir not in sys.path:
    sys.path.insert(0, work_dir)

from ablation.deepvqe_ablation import DeepVQE_Ablation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Tạo model
# KHỞI TẠO MODEL ABLATION (Sửa 'B1a' thành 'B1b' để train biến thể khác)
model = DeepVQE_Ablation.from_config_id('B1a').to(device)

# Bật DataParallel nếu có nhiều GPU
if CONFIG.get('use_multi_gpu', True) and torch.cuda.device_count() > 1:
    print(f'Bật DataParallel trên {torch.cuda.device_count()} GPUs')
    model = torch.nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M')

# Optimizer & Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'],
    min_lr=CONFIG['scheduler_min_lr'],
)

# ★ V2: Mixed Precision GradScaler
use_amp = CONFIG['use_amp'] and device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
print(f'Mixed Precision (AMP): {"ON" if use_amp else "OFF"}')

# STFT window: sqrt-Hann, used consistently for STFT/ISTFT/inference.
window = torch.hann_window(CONFIG['win_length']).sqrt().to(device)

# Output/checkpoint dir
output_dir = Path(CONFIG.get('checkpoint_dir', CONFIG['output_dir']))
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'config.json', 'w') as f:
    json.dump({k: str(v) if isinstance(v, tuple) else v for k, v in CONFIG.items()}, f, indent=2)

# TensorBoard writer is optional; writing to Drive can slow notebook training.
tb_log_dir = output_dir / 'tb_logs'
tb_writer = SummaryWriter(log_dir=str(tb_log_dir)) if CONFIG.get('tensorboard', False) else None
print(f'Output dir: {output_dir}')
print(f'TensorBoard: {"ON" if tb_writer else "OFF"}')
if tb_writer:
    print(f'TensorBoard log: {tb_log_dir}')

## 8. Hàm tiện ích: STFT, Loss, Checkpoint, Log, Metrics
★ V2: SI-SNR tính trên waveform gốc. Thêm hàm PESQ/STOI. Checkpoint lưu scaler state.

In [ ]:
import csv

def make_stft(wav, n_fft, hop_length, win_length, win):
    """wav [B, T] -> spec [B, F, T_frames, 2]"""
    spec = torch.stft(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length,
                      window=win, return_complex=True)
    return torch.view_as_real(spec)

def make_istft(spec, n_fft, hop_length, win_length, win, length=None):
    """spec [B, F, T_frames, 2] -> wav [B, T]"""
    complex_spec = torch.complex(spec[..., 0], spec[..., 1])
    return torch.istft(complex_spec, n_fft=n_fft, hop_length=hop_length,
                       win_length=win_length, window=win, length=length)

def compute_loss(model, noisy_wav, clean_wav, cfg, win):
    """Tính HybridLoss: Compressed RI + Compressed Mag + negative SI-SNR.
    ★ V2: SI-SNR tính trên clean_wav gốc thay vì ISTFT(STFT(clean))."""
    n_fft = cfg['n_fft']
    hop = cfg['hop_length']
    win_len = cfg['win_length']
    c = cfg['compress_factor']
    
    # Forward pass
    noisy_spec = make_stft(noisy_wav, n_fft, hop, win_len, win)
    
    # ★ V2: Chỉ dùng autocast cho model forward pass để tránh lỗi tốc độ với complex STFT
    amp_enabled = cfg.get('use_amp', False) and noisy_wav.device.type == 'cuda'
    with torch.amp.autocast('cuda', enabled=amp_enabled):
        pred_stft = model(noisy_spec)
    
    # Ép kiểu về float32 để tính loss an toàn
    pred_stft = pred_stft.float()
    
    # Cắt padding cho độ dài phù hợp
    clean_spec = make_stft(clean_wav, n_fft, hop, win_len, win)
    min_t = min(pred_stft.shape[2], clean_spec.shape[2])
    pred_stft = pred_stft[:, :, :min_t, :]
    true_stft = clean_spec[:, :, :min_t, :]
    
    # Tách Real / Imaginary
    pred_stft_real, pred_stft_imag = pred_stft[:,:,:,0], pred_stft[:,:,:,1]
    true_stft_real, true_stft_imag = true_stft[:,:,:,0], true_stft[:,:,:,1]
    
    # Tính Magnitude
    pred_mag = torch.sqrt(pred_stft_real**2 + pred_stft_imag**2 + 1e-12)
    true_mag = torch.sqrt(true_stft_real**2 + true_stft_imag**2 + 1e-12)
    
    # Phổ nén (Compressed Spectrum)
    pred_real_c = pred_stft_real / (pred_mag**(1 - c))
    pred_imag_c = pred_stft_imag / (pred_mag**(1 - c))
    true_real_c = true_stft_real / (true_mag**(1 - c))
    true_imag_c = true_stft_imag / (true_mag**(1 - c))
    
    # RI Loss & Mag Loss
    real_loss = torch.mean((pred_real_c - true_real_c)**2)
    imag_loss = torch.mean((pred_imag_c - true_imag_c)**2)
    mag_loss = torch.mean((pred_mag**c - true_mag**c)**2)
    
    # ★ V2: SI-SNR trên waveform — dùng clean_wav GỐC thay vì ISTFT(STFT(clean))
    y_pred = torch.istft(pred_stft_real + 1j*pred_stft_imag, n_fft=n_fft, hop_length=hop, win_length=win_len, window=win)
    y_true = clean_wav  # ★ V2: dùng waveform gốc
    
    # Đảm bảo cùng độ dài
    min_wav_len = min(y_pred.shape[-1], y_true.shape[-1])
    y_pred = y_pred[..., :min_wav_len]
    y_true = y_true[..., :min_wav_len]
    
    eps = 1e-8
    true_energy = torch.sum(torch.square(y_true), dim=-1, keepdim=True)
    y_target = torch.sum(y_true * y_pred, dim=-1, keepdim=True) * y_true / (true_energy + eps)
    target_energy = torch.sum(torch.square(y_target), dim=-1, keepdim=True)
    noise_energy = torch.sum(torch.square(y_pred - y_target), dim=-1, keepdim=True)
    sisnr = -10 * torch.log10((target_energy + eps) / (noise_energy + eps)).mean()  # FIX4: SI-SNR chuẩn dùng 10*log10
    
    # Tổng Loss = 30 * RI + 70 * Mag + SISNR
    loss = cfg['lamda_ri'] * (real_loss + imag_loss) + cfg['lamda_mag'] * mag_loss + sisnr
    
    return loss, {'loss': loss.item(), 'ri_loss': (real_loss + imag_loss).item(), 'mag_loss': mag_loss.item(), 'sisnr': sisnr.item()}


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=None):
    """★ V2: Lưu thêm scaler state cho AMP resume. (Safe Drive Patch)"""
    import os, shutil
    from pathlib import Path
    
    model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    ckpt = {
        'model': model_state,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'best_loss': best_loss,
        'bad_epochs': bad_epochs,
    }
    if scaler is not None:
        ckpt['scaler'] = scaler.state_dict()
        
    base_name = os.path.basename(str(path))
    if os.path.exists('/content'):
        temp_local = f'/content/temp_{base_name}'
    elif os.path.exists('/kaggle/working'):
        temp_local = f'/kaggle/working/temp_{base_name}'
    else:
        temp_local = f'temp_{base_name}'
        
    # 1. Lưu vào SSD cục bộ trước
    torch.save(ckpt, temp_local)
    
    # 2. Copy sang một file tạm trên Drive (.tmp)
    temp_drive = str(path) + '.tmp'
    shutil.copy2(temp_local, temp_drive)
    
    # 3. Ghi đè file cũ bằng file tạm một cách an toàn (atomic rename)
    # Nếu bị ngắt kết nối giữa chừng lúc copy, file gốc vẫn còn nguyên vẹn.
    try:
        os.replace(temp_drive, str(path))
    except OSError:
        # Dự phòng cho một số hệ thống file không hỗ trợ replace trực tiếp
        if os.path.exists(str(path)):
            os.remove(str(path))
        os.rename(temp_drive, str(path))
        
    # 4. Xóa file cục bộ
    if os.path.exists(temp_local):
        os.remove(temp_local)

def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cpu', scaler=None):
    """★ V2: Load scaler state. Backward-compatible với checkpoint V1 (không có scaler)."""
    ckpt = torch.load(str(path), map_location=device, weights_only=False)
    target = model.module if hasattr(model, 'module') else model
    target.load_state_dict(ckpt['model'])
    if optimizer and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler and ckpt.get('scheduler'):
        scheduler.load_state_dict(ckpt['scheduler'])
    if scaler and ckpt.get('scaler'):
        scaler.load_state_dict(ckpt['scaler'])
    return ckpt.get('epoch', 0), ckpt.get('best_loss'), ckpt.get('bad_epochs', 0)


def set_optimizer_lr(optimizer, lr):
    """Đặt LR cho mọi param group."""
    for group in optimizer.param_groups:
        group['lr'] = lr


def optimizer_step_with_amp(optimizer, scaler, model, grad_clip):
    if grad_clip:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)


def append_log(log_path, row_dict):
    log_path = Path(log_path)
    file_exists = log_path.exists() and log_path.stat().st_size > 0
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


def compute_pesq_stoi(model, dataloader, cfg, win, device, max_samples=200):
    """★ V2: Tính PESQ và STOI trung bình trên dataloader.
    max_samples: giới hạn số mẫu để không quá chậm."""
    from pesq import pesq
    from pystoi import stoi

    m = model.module if hasattr(model, 'module') else model
    m.eval()
    sr = cfg['sample_rate']
    pesq_scores, stoi_scores = [], []
    count = 0
    amp_enabled = cfg.get('use_amp', False) and device.type == 'cuda'

    with torch.no_grad():
        for batch in dataloader:
            noisy = batch['noisy'].to(device)
            clean = batch['clean']  # Giữ trên CPU cho pesq/stoi

            spec = make_stft(noisy, cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win)
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                pred = m(spec)
            enhanced = make_istft(pred.float(), cfg['n_fft'], cfg['hop_length'], cfg['win_length'], win)

            for i in range(enhanced.shape[0]):
                enh_np = enhanced[i].cpu().numpy()
                cln_np = clean[i].numpy()
                min_l = min(len(enh_np), len(cln_np))
                enh_np, cln_np = enh_np[:min_l], cln_np[:min_l]

                try:
                    pesq_scores.append(pesq(sr, cln_np, enh_np, 'wb'))
                except Exception:
                    pass  # pesq có thể lỗi trên mẫu quá ngắn
                stoi_scores.append(stoi(cln_np, enh_np, sr, extended=False))

                count += 1
                if count >= max_samples:
                    break
            if count >= max_samples:
                break

    m.train()  # FIX7: restore train mode sau khi eval
    return {
        'pesq': np.mean(pesq_scores) if pesq_scores else 0.0,
        'stoi': np.mean(stoi_scores) if stoi_scores else 0.0,
    }


print('Hàm tiện ích đã sẵn sàng.')


import matplotlib.pyplot as plt
import wandb

def log_audio_to_wandb(model, dataloader, config, device, epoch, window):  # FIX1: nhận window làm tham số
    model.eval()
    sr = config['sample_rate']
    amp_enabled = config.get("use_amp", False) and device.type == "cuda"
    
    with torch.no_grad():
        for batch in dataloader:
            noisy = batch["noisy"].to(device)
            clean = batch["clean"]
            
            spec = make_stft(noisy, config['n_fft'], config['hop_length'], config['win_length'], window)
            with torch.amp.autocast("cuda", enabled=amp_enabled):
                pred = model(spec)
            enhanced = make_istft(pred.float(), config['n_fft'], config['hop_length'], config['win_length'], window, length=noisy.shape[-1])
            
            n_np = noisy[0].cpu().numpy()
            c_np = clean[0].cpu().numpy()
            e_np = enhanced[0].cpu().numpy()
            
            fig, axes = plt.subplots(3, 1, figsize=(10, 8))
            axes[0].specgram(c_np, Fs=sr, NFFT=512, noverlap=256, cmap='viridis')
            axes[0].set_title("Clean")
            axes[1].specgram(n_np, Fs=sr, NFFT=512, noverlap=256, cmap='viridis')
            axes[1].set_title("Noisy")
            axes[2].specgram(e_np, Fs=sr, NFFT=512, noverlap=256, cmap='viridis')
            axes[2].set_title("Enhanced")
            plt.tight_layout()
            
            wandb.log({
                "audio/clean": wandb.Audio(c_np, sample_rate=sr),
                "audio/noisy": wandb.Audio(n_np, sample_rate=sr),
                "audio/enhanced": wandb.Audio(e_np, sample_rate=sr),
                "spectrograms": wandb.Image(fig)
            }, step=epoch)
            plt.close(fig)
            break
    model.train()  # Restore train mode after audio logging


## 8.5 Khởi động TensorBoard (tùy chọn)
Chạy cell này để mở TensorBoard dashboard ngay trong Kaggle. Có thể chạy trước hoặc sau khi bắt đầu training.

In [ ]:
# Mở TensorBoard dashboard nếu CONFIG['tensorboard'] = True.
if CONFIG.get('tensorboard', False):
    %load_ext tensorboard
    %tensorboard --logdir {str(output_dir / 'tb_logs')}
else:
    print('TensorBoard đang tắt trong CONFIG để ưu tiên tốc độ train.')

## 9. Vòng lặp huấn luyện (Training Loop)
- login wandb
- Auto-resume từ `last.pt` nếu có.
- Ghi log mỗi epoch vào `train_log.csv`.
- ★ V2: AMP autocast + GradScaler, gradient accumulation, PESQ/STOI, TensorBoard logging.

In [ ]:
import wandb
from google.colab import userdata

# Lấy khóa từ Secrets của Colab
try:
    my_wandb_key = userdata.get('WANDB_API_KEY') # Đổi tên này nếu bạn đặt tên Secret khác
    wandb.login(key=my_wandb_key)
    print("✅ Đã đăng nhập WandB thành công bằng Secret Key!")
except userdata.SecretNotFoundError:
    print("❌ Không tìm thấy Secret. Hãy kiểm tra lại tên biến trong mục Secrets của Colab.")
    # Dự phòng: Bật khung nhập thủ công nếu không tìm thấy Secret
    wandb.login() 


In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm


if CONFIG.get('use_wandb', False):
    # wandb.login() # User should login via cell beforehand or CLI
    #wandb.init(project=CONFIG.get('wandb_project', 'DeepVQE-Colab-Baseline-K'), config=CONFIG, resume="must", id="ua8mlkzb")
    wandb.init(project=CONFIG.get('wandb_project', 'DeepVQE-Colab-Baseline-K'), config=CONFIG, resume="allow")

    _watch_target = model.module if hasattr(model, 'module') else model  # FIX6: tránh watch DataParallel wrapper
    wandb.watch(_watch_target, log="gradients", log_freq=100)


# === Auto-resume nếu có checkpoint ===
start_epoch = 0
best_loss = None
bad_epochs = 0

resume_path = output_dir / CONFIG.get('resume_checkpoint', 'last.pt')
if resume_path.exists():
    start_epoch, best_loss, bad_epochs = load_checkpoint(
        resume_path, model, optimizer, scheduler, device=device, scaler=scaler  # ★ V2
    )
    resumed_lr = optimizer.param_groups[0]['lr']
    effective_lr = min(resumed_lr, CONFIG['lr'])
    set_optimizer_lr(optimizer, effective_lr)
    _bl_str = f'{best_loss:.6f}' if best_loss is not None else 'N/A'
    print(f'Đã resume từ epoch {start_epoch}, best_loss={_bl_str}, bad_epochs={bad_epochs}')
    print(f'LR checkpoint: {resumed_lr:.2e} | LR config: {CONFIG["lr"]:.2e} -> LR dùng: {effective_lr:.2e}')
else:
    print('Bắt đầu training từ đầu.')
    print(f'LR config: {optimizer.param_groups[0]["lr"]:.2e}')

log_path = output_dir / 'train_log.csv'
print(f'Log file: {log_path}')
print(f'AMP: {"ON" if use_amp else "OFF"} | Batch: {CONFIG["batch_size"]} | Accum: {CONFIG["grad_accum_steps"]}')


# === TRAINING LOOP ===
accum_steps = CONFIG.get('grad_accum_steps', 1)
progress_update_every = max(1, CONFIG.get('progress_update_every', 10))

for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
    # --- TRAIN ---
    model.train()
    train_losses = []
    skipped_train_batches = 0
    valid_accum_batches = 0
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)  # ★ V2: zero_grad trước loop
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
    for batch_idx, batch in enumerate(pbar):
        noisy = batch['noisy'].to(device, non_blocking=True)
        clean = batch['clean'].to(device, non_blocking=True)
        
        # compute_loss only autocasts model forward; STFT/ISTFT/loss stay float32.
        loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
        if not torch.isfinite(loss):
            skipped_train_batches += 1
            optimizer.zero_grad(set_to_none=True)
            print(f'  [WARN] Skip train batch {batch_idx}: non-finite loss | ids={batch["id"][:3]}')
            continue
        loss = loss / accum_steps
        
        scaler.scale(loss).backward()
        
        valid_accum_batches += 1
        # ★ V2: Gradient accumulation — chỉ step optimizer sau mỗi accum_steps batch hợp lệ
        if valid_accum_batches % accum_steps == 0:
            optimizer_step_with_amp(optimizer, scaler, model, CONFIG['grad_clip'])
        
        train_losses.append(parts)
        if batch_idx % progress_update_every == 0 or (batch_idx + 1) == len(train_loader):
            pbar.set_postfix({'loss': f"{parts['loss']:.4f}"})
    
    if valid_accum_batches % accum_steps != 0:
        optimizer_step_with_amp(optimizer, scaler, model, CONFIG['grad_clip'])
    if not train_losses:
        print('Không có train batch hợp lệ trong epoch này; dừng training để tránh ghi checkpoint lỗi.')
        break
    avg_train = {k: np.mean([d[k] for d in train_losses]) for k in train_losses[0]}
    elapsed = time.time() - t0
    
    # --- VALID ---
    model.eval()
    valid_losses = []
    skipped_valid_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f'Epoch {epoch} [Valid]', leave=False):
            noisy = batch['noisy'].to(device, non_blocking=True)
            clean = batch['clean'].to(device, non_blocking=True)
            valid_loss, parts = compute_loss(model, noisy, clean, CONFIG, window)
            if not torch.isfinite(valid_loss):
                skipped_valid_batches += 1
                print(f'  [WARN] Skip valid batch: non-finite loss | ids={batch["id"][:3]}')
                continue
            valid_losses.append(parts)
    
    if not valid_losses:
        print('Không có valid batch hợp lệ trong epoch này; dừng training để tránh ghi checkpoint lỗi.')
        break
    avg_valid = {k: np.mean([d[k] for d in valid_losses]) for k in valid_losses[0]}
    train_has_nan = skipped_train_batches > 0 or not np.isfinite(avg_train['loss'])
    valid_has_nan = skipped_valid_batches > 0 or not np.isfinite(avg_valid['loss'])
    
    # ★ V2: Perceptual Metrics (mỗi N epoch)
    pesq_stoi = {'pesq': 0.0, 'stoi': 0.0}
    eval_every = CONFIG.get('eval_pesq_every', 0)
    if eval_every > 0 and epoch % eval_every == 0:
        print(f'  Đang tính PESQ/STOI (mỗi {eval_every} epoch)...')
        pesq_stoi = compute_pesq_stoi(model, valid_loader, CONFIG, window, device)
        print(f'  >> PESQ: {pesq_stoi["pesq"]:.3f} | STOI: {pesq_stoi["stoi"]:.4f}')
    
    if train_has_nan or valid_has_nan:
        # FIX5: Chỉ warn, không break — training vẫn tiếp tục nhưng bỏ qua scheduler/checkpoint epoch này
        print(f'  [WARN] Epoch {epoch} có batch non-finite: train_skipped={skipped_train_batches}, valid_skipped={skipped_valid_batches}.')
        print('  [WARN] Bỏ qua scheduler/checkpoint epoch này. Training vẫn tiếp tục...')
        continue
    
    # --- Scheduler ---
    prev_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_valid['loss'])
    current_lr = optimizer.param_groups[0]['lr']
    scheduler_bad_epochs = getattr(scheduler, 'num_bad_epochs', None)
    scheduler_best = getattr(scheduler, 'best', None)
    if current_lr < prev_lr:
        print(f'  >>> Scheduler giảm LR: {prev_lr:.2e} -> {current_lr:.2e}')
    
    # --- Logging ---
    print(
        f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
        f"Train Loss: {avg_train['loss']:.6f} (ri={avg_train['ri_loss']:.4f}, mag={avg_train['mag_loss']:.4f}, sisnr={avg_train['sisnr']:.4f}) | "
        f"Valid Loss: {avg_valid['loss']:.6f} (ri={avg_valid['ri_loss']:.4f}, mag={avg_valid['mag_loss']:.4f}, sisnr={avg_valid['sisnr']:.4f}) | "
        f"LR: {current_lr:.2e} | Sched bad: {scheduler_bad_epochs}/{CONFIG['scheduler_patience']} | Time: {elapsed:.0f}s"
    )
    
    # --- Ghi log vào CSV ---
    append_log(log_path, {
        'epoch': epoch,
        'train_loss': f"{avg_train['loss']:.6f}",
        'train_ri_loss': f"{avg_train['ri_loss']:.6f}",
        'train_mag_loss': f"{avg_train['mag_loss']:.6f}",
        'train_sisnr': f"{avg_train['sisnr']:.6f}",
        'valid_loss': f"{avg_valid['loss']:.6f}",
        'valid_ri_loss': f"{avg_valid['ri_loss']:.6f}",
        'valid_mag_loss': f"{avg_valid['mag_loss']:.6f}",
        'valid_sisnr': f"{avg_valid['sisnr']:.6f}",
        'valid_pesq': f"{pesq_stoi['pesq']:.4f}",      # ★ V2
        'valid_stoi': f"{pesq_stoi['stoi']:.4f}",      # ★ V2
        'lr': f"{current_lr:.2e}",
        'scheduler_bad_epochs': scheduler_bad_epochs,
        'time_s': f"{elapsed:.0f}",
    })
    
    # TensorBoard logging (optional; can be slow when logdir is on Drive)
    if tb_writer:
        tb_writer.add_scalars('Loss/Total', {'train': avg_train['loss'], 'valid': avg_valid['loss']}, epoch)
        tb_writer.add_scalars('Loss/RI', {'train': avg_train['ri_loss'], 'valid': avg_valid['ri_loss']}, epoch)
        tb_writer.add_scalars('Loss/Mag', {'train': avg_train['mag_loss'], 'valid': avg_valid['mag_loss']}, epoch)
        tb_writer.add_scalars('Loss/SISNR', {'train': avg_train['sisnr'], 'valid': avg_valid['sisnr']}, epoch)
        tb_writer.add_scalar('LR', current_lr, epoch)
        if pesq_stoi['pesq'] > 0:
            tb_writer.add_scalar('Metrics/PESQ', pesq_stoi['pesq'], epoch)
            tb_writer.add_scalar('Metrics/STOI', pesq_stoi['stoi'], epoch)
        tb_writer.flush()
    # --- WandB Logging ---
    if CONFIG.get('use_wandb', False) and wandb.run is not None:
        wandb_metrics = {
            'epoch': epoch,
            'train/loss': avg_train['loss'],
            'train/ri_loss': avg_train['ri_loss'],
            'train/mag_loss': avg_train['mag_loss'],
            'train/sisnr': avg_train['sisnr'],
            'valid/loss': avg_valid['loss'],
            'valid/ri_loss': avg_valid['ri_loss'],
            'valid/mag_loss': avg_valid['mag_loss'],
            'valid/sisnr': avg_valid['sisnr'],
            'lr': current_lr,
        }
        if pesq_stoi.get('pesq', 0) > 0:
            wandb_metrics['metrics/pesq'] = pesq_stoi['pesq']
            wandb_metrics['metrics/stoi'] = pesq_stoi['stoi']
            
        wandb.log(wandb_metrics, step=epoch)

        log_audio_every = CONFIG.get('log_audio_every', 0)
        if log_audio_every > 0 and epoch % log_audio_every == 0:
            log_audio_to_wandb(model, valid_loader, CONFIG, device, epoch, window)  # FIX1b: truyền window


    
    # --- Checkpoint ---
    is_best = best_loss is None or avg_valid['loss'] < best_loss
    if is_best:
        best_loss = avg_valid['loss']
        bad_epochs = 0
    else:
        bad_epochs += 1
    
    # Luôn lưu last.pt
    save_checkpoint(output_dir / 'last.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler)  # ★ V2
    
    # Lưu best.pt nếu tốt hơn
    if is_best:
        save_checkpoint(output_dir / 'best.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler)  # ★ V2
        print(f'  >>> Saved best model (valid_loss={best_loss:.6f})')
    
    # Early stopping
    if bad_epochs >= CONFIG['early_stopping_patience']:
        print(f'Early stopping sau {bad_epochs} epoch không cải thiện. Best loss: {best_loss:.6f}')
        break

if tb_writer:
    tb_writer.close()
print(f'\n=== Huấn luyện hoàn tất! ===')
print(f'Best valid loss: {best_loss:.6f}')
print(f'Checkpoint: {output_dir}')
if tb_writer:
    print(f'TensorBoard log: {tb_log_dir}')


## 10. Kiểm tra nhanh sau training
Chạy inference trên một mẫu test để xác nhận model hoạt động.

In [ ]:
import IPython.display as ipd

# Load best model
best_ckpt = output_dir / 'best.pt'
if best_ckpt.exists():
    load_checkpoint(best_ckpt, model, device=device)
    print('Đã load best.pt')

model.eval()
test_df = pd.read_csv(CONFIG['test_csv'])
sample = test_df.iloc[0]

sig_noisy, sr = torchaudio.load(sample['noisy_wav'])
if sr != CONFIG['sample_rate']:
    sig_noisy = torchaudio.functional.resample(sig_noisy, sr, CONFIG['sample_rate'])

with torch.no_grad():
    noisy_wav = sig_noisy.squeeze(0).to(device)
    spec = make_stft(noisy_wav.unsqueeze(0), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
    # Xử lý DataParallel
    m = model.module if hasattr(model, 'module') else model
    with torch.amp.autocast('cuda', enabled=use_amp):  # ★ V2
        pred = m(spec)
    enhanced = make_istft(pred.float(), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window, length=noisy_wav.shape[0])

print(f'Sample: {sample["ID"]}')
print('Noisy:')
ipd.display(ipd.Audio(sig_noisy.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))
print('Enhanced:')
ipd.display(ipd.Audio(enhanced.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))

sig_clean, sr_c = torchaudio.load(sample['clean_wav'])
if sr_c != CONFIG['sample_rate']:
    sig_clean = torchaudio.functional.resample(sig_clean, sr_c, CONFIG['sample_rate'])
print('Clean (ground truth):')
ipd.display(ipd.Audio(sig_clean.squeeze().cpu().numpy(), rate=CONFIG['sample_rate']))

## 11. Chạy ablation benchmarks trên Kaggle
Cell này chạy smoke checks, benchmark kiến trúc cho các cấu hình trong `ablation/`, và có thể bật thêm train/eval/ONNX khi cần.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== TÙY CHỈNH NHANH =====================
WORKSPACE = Path('/kaggle/working/DeepVQE_Workspace')
AB_SOURCE_DIR = None  # ví dụ: '/content/drive/MyDrive/deepvqe/ablation'; None = tự tìm/clone Pdeepvqe
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/content/Pdeepvqe_benchmark_src')

# Nếu muốn chạy ít cấu hình để smoke test trước, sửa thành: ['Baseline', 'B1a']
AB_CONFIGS = [
    'Baseline', 'B1a', 'B1b', 'B2', 'B3',
    'C1', 'C1a-g2', 'C1a-g4', 'C1b', 'C2', 'C3', 'C4',
]

RUN_SMOKE_TESTS = True
RUN_ARCH_BENCHMARK = True

# None = tự tìm best.pt trong Drive/Colab. Nếu có nhiều best.pt, điền path cụ thể vào đây.
BASELINE_CHECKPOINT = None

# Hai phần dưới tốn GPU lâu hơn. Bật True khi muốn benchmark chất lượng sau train.
RUN_TRAINING = False
RUN_QUALITY_EVAL = True
RUN_ONNX_EXPORT = False

AB_EPOCHS = 1
AB_BATCH_SIZE = 8
AB_NUM_WORKERS = 0
AB_PIN_MEMORY = False
AB_EARLY_STOP_PATIENCE = 12
AB_EARLY_STOP_MIN_DELTA = 0.01
AB_EARLY_STOP_MIN_EPOCHS = 20
AB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AB_USE_DATA_PARALLEL = CONFIG.get('use_multi_gpu', True) and torch.cuda.is_available() and torch.cuda.device_count() > 1
RESUME_EXISTING = True

ARCH_FRAMES = 63
ARCH_WARMUP = 1
ARCH_REPEATS = 3
INSTALL_OPTIONAL_PACKAGES = False  # ptflops / pesq / pystoi / onnxruntime nếu cần
# ===========================================================

repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = WORKSPACE / 'deepvqe'
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

print(f'Repo dir: {repo_dir}')
print(f'Device: {AB_DEVICE}')
if torch.cuda.is_available():
    print(f'CUDA GPUs: {torch.cuda.device_count()}')
    for gpu_idx in range(torch.cuda.device_count()):
        print(f'  GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}')
print(f'DataParallel: {AB_USE_DATA_PARALLEL}')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source(explicit_dir=None):
    if explicit_dir:
        path = Path(explicit_dir)
        if not (path / 'deepvqe_ablation.py').exists():
            raise FileNotFoundError(f'AB_SOURCE_DIR không hợp lệ: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    for root in roots:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if BENCHMARK_REPO_DIR.exists() and (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'deepvqe_ablation.py').exists():
    source_ablation = find_ablation_source(AB_SOURCE_DIR)
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/deepvqe_ablation.py. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, ablation streaming benchmark sẽ không chạy được.')

def patch_eval_script_for_uploaded_checkpoints():
    path = repo_dir / 'ablation' / 'eval_ablation_quality.py'
    if not path.exists():
        return
    text = path.read_text(encoding='utf-8')
    required = ['torch_load_checkpoint', 'load_state_dict_flexible', 'make_window']
    missing = [name for name in required if name not in text]
    if missing:
        raise RuntimeError(f'{path} thiếu benchmark checkpoint/window helpers: {missing}. Hãy cập nhật repo trước khi eval.')
    print(f'{path} đã hỗ trợ checkpoint flexible và STFT window đúng config.')

patch_eval_script_for_uploaded_checkpoints()

results_dir = WORKSPACE / 'results/ablation'
runs_dir = WORKSPACE / 'runs/ablation'
onnx_dir = WORKSPACE / 'onnx_models/ablation'
manifest_dir = WORKSPACE / 'metadata/ablation_manifests'
for path in [results_dir, runs_dir, onnx_dir, manifest_dir]:
    path.mkdir(parents=True, exist_ok=True)

arch_csv = results_dir / 'ablation_arch_benchmark.csv'
quality_csv = results_dir / 'ablation_quality.csv'
onnx_csv = results_dir / 'ablation_onnx.csv'
summary_csv = results_dir / 'ablation_summary.csv'

def find_best_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy BASELINE_CHECKPOINT: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob('best.pt'))
            if candidates:
                break
    if not candidates:
        return None

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

uploaded_baseline_ckpt = find_best_checkpoint(BASELINE_CHECKPOINT)
if uploaded_baseline_ckpt is not None:
    print(f'Dùng Baseline checkpoint: {uploaded_baseline_ckpt}')
else:
    print('Không tìm thấy best.pt trong Drive/Colab; Baseline eval/ONNX sẽ dùng checkpoint trong runs_dir nếu có.')

def checkpoint_for_config(config_id):
    if config_id == 'Baseline' and uploaded_baseline_ckpt is not None:
        return uploaded_baseline_ckpt
    return runs_dir / config_id / 'best.pt'

if INSTALL_OPTIONAL_PACKAGES:
    packages = ['ptflops']
    if RUN_QUALITY_EVAL:
        packages += ['pesq', 'pystoi']
    if RUN_ONNX_EXPORT:
        packages += ['onnx', 'onnxruntime', 'onnxsim']
    run_py(['-m', 'pip', 'install', '-q', *packages])

def csv_to_jsonl(csv_path, jsonl_path):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path

metadata_base = Path(globals().get('data_dir', str(WORKSPACE / 'data/voicebank-demand')))
csv_manifests = {
    'train': metadata_base / 'train.csv',
    'valid': metadata_base / 'valid.csv',
    'test': metadata_base / 'test.csv',
}
jsonl_manifests = {}
if all(path.exists() for path in csv_manifests.values()):
    for split, csv_path in csv_manifests.items():
        jsonl_manifests[split] = csv_to_jsonl(csv_path, manifest_dir / f'{split}.jsonl')
    print('Đã tạo manifest JSONL cho ablation:')
    for split, path in jsonl_manifests.items():
        print(f'  {split}: {path}')
elif RUN_TRAINING or RUN_QUALITY_EVAL:
    raise FileNotFoundError('Thiếu train/valid/test CSV. Hãy chạy cell tạo metadata trước.')
else:
    print('Chưa có metadata CSV; bỏ qua manifest train/eval và chỉ chạy benchmark kiến trúc.')

if RUN_SMOKE_TESTS:
    run_py(['ablation/test_ablation_baseline.py'])
    run_py(['ablation/test_ablation_streaming.py', '--configs', *AB_CONFIGS])

if RUN_ARCH_BENCHMARK:
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', arch_csv,
        '--configs', *AB_CONFIGS,
        '--device', AB_DEVICE,
        '--frames', ARCH_FRAMES,
        '--warmup', ARCH_WARMUP,
        '--repeats', ARCH_REPEATS,
    ])

if RUN_TRAINING:
    for config_id in AB_CONFIGS:
        ab_output_dir = runs_dir / config_id
        args = [
            'ablation/train_ablation.py',
            '--config-id', config_id,
            '--train-manifest', jsonl_manifests['train'],
            '--valid-manifest', jsonl_manifests['valid'],
            '--output-dir', ab_output_dir,
            '--device', AB_DEVICE,
            '--epochs', AB_EPOCHS,
            '--batch-size', AB_BATCH_SIZE,
            '--num-workers', AB_NUM_WORKERS,
            '--early-stop-patience', AB_EARLY_STOP_PATIENCE,
            '--early-stop-min-delta', AB_EARLY_STOP_MIN_DELTA,
            '--early-stop-min-epochs', AB_EARLY_STOP_MIN_EPOCHS,
        ]
        last_ckpt = ab_output_dir / 'last.pt'
        if not AB_PIN_MEMORY:
            args += ['--no-pin-memory']
        if AB_USE_DATA_PARALLEL:
            args += ['--data-parallel']
        if RESUME_EXISTING and last_ckpt.exists():
            args += ['--resume', last_ckpt, '--ignore-bad-resume']
        run_py(args)

if RUN_QUALITY_EVAL:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua eval {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/eval_ablation_quality.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--manifest', jsonl_manifests['test'],
            '--output', quality_csv,
            '--device', AB_DEVICE,
        ])

if RUN_ONNX_EXPORT:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua ONNX {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/export_ablation_onnx.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--output-dir', onnx_dir,
            '--results', onnx_csv,
            '--device', 'cpu',
        ])

run_py([
    'ablation/collect_ablation_results.py',
    '--arch', arch_csv,
    '--quality', quality_csv,
    '--onnx', onnx_csv,
    '--output', summary_csv,
])

print('\nCSV kết quả:')
for csv_path in [arch_csv, quality_csv, onnx_csv, summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path).head(30))

archive_base = Path('/content/ablation_benchmark_results')
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(results_dir))
print(f'Đã nén kết quả benchmark: {archive_path}')
display(FileLink(str(archive_path)))

## 12. Đánh giá chất lượng mô hình: PESQ, STOI, SI-SDR, RTF
Đo lường các chỉ số chất lượng âm thanh sau khi đi qua mô hình:
- **PESQ (Perceptual Evaluation of Speech Quality)**: Đánh giá chất lượng giọng nói cảm nhận (điểm càng cao càng tốt, tối đa 4.5).
- **STOI (Short-Time Objective Intelligibility)**: Đánh giá độ rõ (hiểu được) của giọng nói (điểm từ 0 đến 1, càng cao càng tốt).
- **SI-SDR (Scale-Invariant Signal-to-Distortion Ratio)**: Đánh giá tỷ lệ tín hiệu trên nhiễu (điểm càng cao càng tốt).
- **RTF (Real-Time Factor)**: Tốc độ xử lý (thời gian xử lý / thời gian audio). RTF < 1 nghĩa là xử lý nhanh hơn thời gian thực.

In [ ]:
!pip install -q pesq pystoi torchmetrics pandas IPython tqdm

import time
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
import warnings
from pesq import pesq
from pystoi import stoi
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio
from IPython.display import display, FileLink
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# --- Load best model nếu chưa load ---
best_ckpt_path = output_dir / 'best.pt'
if best_ckpt_path.exists():
    print(f"Tải trọng số tốt nhất từ: {best_ckpt_path}")
    checkpoint = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    # Lấy model module nếu bọc trong DataParallel
    target_model = model.module if hasattr(model, 'module') else model
    target_model.load_state_dict(checkpoint['model'])
    
model.eval()

# Đếm tham số
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {total_params:,} (trainable: {trainable_params:,})')
print(f'Device: {device}')

# --- Load test set ---
test_csv_path = CONFIG.get('test_csv', f'{data_dir}/test.csv')
print(f'Test CSV: {test_csv_path}')
df_test = pd.read_csv(test_csv_path)

# (Tùy chọn) Rút gọn test_set nếu cần test nhanh
EVAL_MAX_SAMPLES = None # Đổi thành số lượng (vd: 50) để chạy nhanh
if EVAL_MAX_SAMPLES is not None:
    df_test = df_test.head(EVAL_MAX_SAMPLES)
print(f'Test samples: {len(df_test)}')

# Khởi tạo SI-SDR metric từ torchmetrics
sisdr_metric = ScaleInvariantSignalDistortionRatio().to(device)

results = []
total_audio_duration = 0.0
total_processing_time = 0.0
EVAL_SR = CONFIG.get('sample_rate', 16000)

with torch.no_grad():
    for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc='Evaluating'):
        # Load audio từ đường dẫn
        noisy_wav, sr_n = torchaudio.load(row['noisy_wav'])
        clean_wav, sr_c = torchaudio.load(row['clean_wav'])
        
        # Chuyển thành mono nếu nhiều kênh
        if noisy_wav.shape[0] > 1:
            noisy_wav = noisy_wav.mean(dim=0, keepdim=True)
        if clean_wav.shape[0] > 1:
            clean_wav = clean_wav.mean(dim=0, keepdim=True)
            
        # Resample nếu cần
        if sr_n != EVAL_SR:
            noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, EVAL_SR)
        if sr_c != EVAL_SR:
            clean_wav = torchaudio.functional.resample(clean_wav, sr_c, EVAL_SR)
        
        noisy_wav = noisy_wav.squeeze(0)  # [T]
        clean_wav = clean_wav.squeeze(0)  # [T]
        
        audio_duration = noisy_wav.shape[0] / EVAL_SR
        total_audio_duration += audio_duration
        
        # === Inference với đo RTF ===
        noisy_gpu = noisy_wav.unsqueeze(0).to(device) # Thêm batch dim [1, T]
        clean_gpu = clean_wav.unsqueeze(0).to(device)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t_start = time.perf_counter()
        
        # Pipeline y hệt trong training (make_stft, model, make_istft)
        noisy_spec = make_stft(noisy_gpu, CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
        
        amp_enabled = CONFIG.get('use_amp', False) and device.type == 'cuda'
        with torch.amp.autocast('cuda', enabled=amp_enabled):
            pred_stft = model(noisy_spec)
        
        pred_stft = pred_stft.float()
        enhanced = make_istft(pred_stft, CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window, length=noisy_gpu.shape[-1])
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t_end = time.perf_counter()
        
        processing_time = t_end - t_start
        total_processing_time += processing_time
        rtf = processing_time / audio_duration
        
        # Tính SI-SDR trên GPU
        si_sdr_enhanced = sisdr_metric(enhanced, clean_gpu).item()
        si_sdr_noisy = sisdr_metric(noisy_gpu, clean_gpu).item()
        
        # Chuyển về Numpy CPU cho PESQ và STOI
        enhanced_np = enhanced.squeeze(0).cpu().numpy()
        clean_np = clean_wav.numpy()
        noisy_np = noisy_wav.numpy()
        
        # Cắt bằng độ dài nhau
        min_len = min(len(enhanced_np), len(clean_np))
        enhanced_np = enhanced_np[:min_len]
        clean_np = clean_np[:min_len]
        noisy_np = noisy_np[:min_len]
        
        # PESQ
        try:
            pesq_enhanced = pesq(EVAL_SR, clean_np, enhanced_np, 'wb')
            pesq_noisy = pesq(EVAL_SR, clean_np, noisy_np, 'wb')
        except Exception:
            pesq_enhanced = float('nan')
            pesq_noisy = float('nan')
            
        # STOI
        try:
            stoi_enhanced = stoi(clean_np, enhanced_np, EVAL_SR, extended=False)
            stoi_noisy = stoi(clean_np, noisy_np, EVAL_SR, extended=False)
        except Exception:
            stoi_enhanced = float('nan')
            stoi_noisy = float('nan')
            
        results.append({
            'ID': row.get('ID', f'Sample_{idx}'),
            'PESQ_enhanced': round(pesq_enhanced, 4),
            'PESQ_noisy': round(pesq_noisy, 4),
            'PESQ_improvement': round(pesq_enhanced - pesq_noisy, 4) if not (np.isnan(pesq_enhanced) or np.isnan(pesq_noisy)) else float('nan'),
            'STOI_enhanced': round(stoi_enhanced, 4),
            'STOI_noisy': round(stoi_noisy, 4),
            'STOI_improvement': round(stoi_enhanced - stoi_noisy, 4) if not (np.isnan(stoi_enhanced) or np.isnan(stoi_noisy)) else float('nan'),
            'SI_SDR_enhanced_dB': round(si_sdr_enhanced, 2),
            'SI_SDR_noisy_dB': round(si_sdr_noisy, 2),
            'SI_SDR_improvement_dB': round(si_sdr_enhanced - si_sdr_noisy, 2),
            'RTF': round(rtf, 6),
            'duration_s': round(audio_duration, 3),
        })

# --- Tổng hợp kết quả ---
df_results = pd.DataFrame(results)

# Aggregate stats
overall_rtf = total_processing_time / total_audio_duration
summary = {
    'Metric': ['PESQ (enhanced)', 'PESQ (noisy)', 'PESQ (Δ improvement)',
               'STOI (enhanced)', 'STOI (noisy)', 'STOI (Δ improvement)',
               'SI-SDR enhanced (dB)', 'SI-SDR noisy (dB)', 'SI-SDR Δ (dB)',
               'RTF (mean)', 'RTF (overall)', 'Real-time capable?'],
    'Value': [
        f"{df_results['PESQ_enhanced'].mean():.4f}",
        f"{df_results['PESQ_noisy'].mean():.4f}",
        f"{df_results['PESQ_improvement'].mean():.4f}",
        f"{df_results['STOI_enhanced'].mean():.4f}",
        f"{df_results['STOI_noisy'].mean():.4f}",
        f"{df_results['STOI_improvement'].mean():.4f}",
        f"{df_results['SI_SDR_enhanced_dB'].mean():.2f}",
        f"{df_results['SI_SDR_noisy_dB'].mean():.2f}",
        f"{df_results['SI_SDR_improvement_dB'].mean():.2f}",
        f"{df_results['RTF'].mean():.6f}",
        f"{overall_rtf:.6f}",
        '✅ Có' if overall_rtf < 1.0 else '❌ Không',
    ]
}
df_summary = pd.DataFrame(summary)

print('\n' + '=' * 60)
print('  KẾT QUẢ ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH DeepVQE (V4)')  # FIX9: đúng version
print('=' * 60)
print(f'Test samples: {len(df_results)}')
print(f'Model params: {total_params:,}')
print(f'Device: {device}')
print(f'Total audio: {total_audio_duration:.1f}s | Processing: {total_processing_time:.1f}s')
print()
display(df_summary)

print('\n--- Chi tiết 10 mẫu đầu tiên ---')
display(df_results.head(10))

# --- Lưu CSV ---
EVAL_SAVE_CSV = True
if EVAL_SAVE_CSV:
    from pathlib import Path
    # Lưu vào output_dir (đã được định nghĩa ở trên như f'{SHARED_CHECKPOINTS_DIR}/deepvqe_vctk')
    save_dir = output_dir / 'evaluation'
    save_dir.mkdir(parents=True, exist_ok=True)
    
    detail_csv = save_dir / 'eval_metrics_per_sample.csv'
    summary_csv_path = save_dir / 'eval_metrics_summary.csv'
    
    df_results.to_csv(detail_csv, index=False)
    df_summary.to_csv(summary_csv_path, index=False)
    
    print(f'\n📁 Đã lưu kết quả chi tiết:  {detail_csv}')
    print(f'📁 Đã lưu bảng tổng hợp:     {summary_csv_path}')
    
    display(FileLink(str(detail_csv)))
    display(FileLink(str(summary_csv_path)))
